In [134]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import ast
from collections import Counter

In [135]:
telemetry_dir_name = 'telemetry20251017_132454'

flow_stats_path = f"../{telemetry_dir_name}/flow_stats.csv"
port_stats_path = f"../{telemetry_dir_name}/port_stats.csv"
table_stats_path = f"../{telemetry_dir_name}/table_stats.csv"
alerts_path = f"../{telemetry_dir_name}/alerts.csv"
log_file_path = f"../{telemetry_dir_name}/traffic_simulation.log"

output_path = "ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks.csv"

In [136]:
def parse_simulation_start_end(log_file_path):
    """
    Parse the overall simulation start and end times from traffic_simulation.log.
    Returns (start_time, end_time) as pandas Timestamps.
    """
    start_time = None
    end_time = None
    
    with open(log_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            
            start_match = re.search(r'Start Time:\s*(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', line)
            if start_match:
                start_time = pd.to_datetime(start_match.group(1))
            
            end_match = re.search(r'End Time:\s*(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', line)
            if end_match:
                end_time = pd.to_datetime(end_match.group(1))
    
    if start_time is None or end_time is None:
        raise ValueError("Could not find simulation start/end times in log.")
    
    return start_time, end_time

In [137]:
start_time, end_time = parse_simulation_start_end(log_file_path)
print(f"Start time: {start_time}\nEnd time: {end_time}")

Start time: 2025-10-17 15:25:46
End time: 2025-10-17 16:26:15


In [138]:
def parse_alert_line(alert_line):
    """
    Parse alert line to extract timestamp and attack type.
    Format: '10/17-15:25:46.605939,1,1000002,1,Possible SYN flood - many SYNs to dst,TCP,...'
    Returns: (timestamp, attack_type)
    """
    if pd.isna(alert_line) or not isinstance(alert_line, str):
        return pd.NaT, None
    
    try:
        # Split by tabs
        parts = alert_line.split(',')
        if len(parts) < 5:
            return pd.NaT, None
        
        # Extract timestamp (first column)
        timestamp_str = parts[0].strip()
        month_day, time_part = timestamp_str.split('-')
        month, day = month_day.split('/')
        dt_str = f"2025-{month.zfill(2)}-{day.zfill(2)} {time_part}"
        timestamp = pd.to_datetime(dt_str, format='%Y-%m-%d %H:%M:%S.%f')
        
        # Extract attack type from message (5th column)
        message = parts[4].strip().lower()
        attack_type = None
        if 'syn flood' in message:
            attack_type = 'syn'
        elif 'udp flood' in message:
            attack_type = 'udp'
        elif 'icmp flood' in message:
            attack_type = 'icmp'
        
        return timestamp, attack_type
    except Exception as e:
        print(f"Warning: Could not parse alert line '{alert_line}': {e}")
        return pd.NaT, None

In [139]:
def parse_stats_timestamp(ts):
    """Parse stats timestamp format 'Sun Oct 12 13:54:29 2025'"""
    if pd.isna(ts):
        return pd.NaT
    try:
        return pd.to_datetime(ts, format='%a %b %d %H:%M:%S %Y')
    except Exception as e:
        print(f"Warning: Could not parse stats timestamp '{ts}': {e}")
        return pd.NaT

In [140]:
def extract_match_fields(match_str):
    """Extract fields from OFPMatch string"""
    if pd.isna(match_str) or match_str == '':
        return {}
    
    try:
        match = re.search(r'oxm_fields=({.*})', str(match_str))
        if match:
            dict_str = match.group(1)
            return ast.literal_eval(dict_str)
    except:
        pass
    return {}

def get_protocol_from_match(match_dict):
    """Determine protocol type from match fields"""
    if 'ip_proto' in match_dict:
        proto = match_dict['ip_proto']
        if proto == 6:
            return 'tcp'
        elif proto == 17:
            return 'udp'
        elif proto == 1:
            return 'icmp'
    return 'other'

In [141]:
def create_ml_dataset(flow_stats_path, port_stats_path, table_stats_path, alerts_path):
    """Create ML dataset from SDN statistics and Snort alerts with multi-attack support"""

    # Load datasets
    flow_df = pd.read_csv(flow_stats_path)
    port_df = pd.read_csv(port_stats_path)
    table_df = pd.read_csv(table_stats_path)
    
    # Load alerts - read the full line
    with open(alerts_path, 'r') as f:
        alert_lines = [line.strip() for line in f if line.strip()]
    
    # Parse each alert line to extract timestamp and attack type
    print("Parsing alerts...")
    parsed_alerts = [parse_alert_line(line) for line in alert_lines]
    alerts_df = pd.DataFrame(parsed_alerts, columns=['timestamp', 'attack_type'])
    
    # Remove NaT values and unknown attack types
    print(f"Alerts before filtering: {len(alerts_df)}")
    alerts_df = alerts_df[alerts_df['timestamp'].notna() & alerts_df['attack_type'].notna()]
    print(f"Alerts after filtering: {len(alerts_df)}")
    print(f"Attack type distribution:\n{alerts_df['attack_type'].value_counts()}")

    # Parse timestamps for stats
    print("Parsing timestamps...")
    flow_df['timestamp'] = flow_df['ts'].apply(parse_stats_timestamp)
    port_df['timestamp'] = port_df['ts'].apply(parse_stats_timestamp)
    table_df['timestamp'] = table_df['ts'].apply(parse_stats_timestamp)

    print(f"\nFiltering data to time range: {start_time} to {end_time}")
    flow_df = flow_df[(flow_df['timestamp'] >= start_time) & (flow_df['timestamp'] <= end_time)]
    port_df = port_df[(port_df['timestamp'] >= start_time) & (port_df['timestamp'] <= end_time)]
    table_df = table_df[(table_df['timestamp'] >= start_time) & (table_df['timestamp'] <= end_time)]
    alerts_df = alerts_df[(alerts_df['timestamp'] >= start_time) & (alerts_df['timestamp'] <= end_time)]
    
    print(f"Rows after filtering - Flow: {len(flow_df)}, Port: {len(port_df)}, Table: {len(table_df)}, Alerts: {len(alerts_df)}")
    
    # Round timestamps to seconds for aggregation
    flow_df['time_bucket'] = flow_df['timestamp'].dt.floor('1s')
    port_df['time_bucket'] = port_df['timestamp'].dt.floor('1s')
    table_df['time_bucket'] = table_df['timestamp'].dt.floor('1s')
    alerts_df['time_bucket'] = alerts_df['timestamp'].dt.floor('1s')
    
    # Create attack type mapping per time bucket
    # For each time bucket, determine which attack types are present
    attack_map = {}
    for _, row in alerts_df.iterrows():
        tb = row['time_bucket']
        at = row['attack_type']
        if tb not in attack_map:
            attack_map[tb] = set()
        attack_map[tb].add(at)
    
    print(f"\nTime buckets with attacks: {len(attack_map)}")
    
    # Extract match fields from flow stats
    print("Extracting flow match fields...")
    flow_df['match_dict'] = flow_df['match'].apply(extract_match_fields)
    flow_df['protocol'] = flow_df['match_dict'].apply(get_protocol_from_match)
    flow_df['src_ip'] = flow_df['match_dict'].apply(lambda x: x.get('ipv4_src', None))
    flow_df['dst_ip'] = flow_df['match_dict'].apply(lambda x: x.get('ipv4_dst', None))
    flow_df['src_port'] = flow_df['match_dict'].apply(lambda x: x.get('tcp_src', x.get('udp_src', None)))
    flow_df['dst_port'] = flow_df['match_dict'].apply(lambda x: x.get('tcp_dst', x.get('udp_dst', None)))
    
    # Get all unique time buckets
    all_times = sorted(set(flow_df['time_bucket']) | set(port_df['time_bucket']) | set(table_df['time_bucket']))
    
    print(f"Processing {len(all_times)} time windows...")
    
    dataset_rows = []
    
    for i, time_bucket in enumerate(all_times):
        if i % 100 == 0:
            print(f"Processing {i}/{len(all_times)}...")
        
        row = {'timestamp': time_bucket}
        
        # Get data for current time bucket
        flow_window = flow_df[flow_df['time_bucket'] == time_bucket]
        port_window = port_df[port_df['time_bucket'] == time_bucket]
        table_window = table_df[table_df['time_bucket'] == time_bucket]
                
        # Flow statistics aggregation
        if len(flow_window) > 0:
            row['flow_packet_count_sum'] = flow_window['packet_count'].sum()
            row['flow_packet_count_mean'] = flow_window['packet_count'].mean()
            row['flow_packet_count_std'] = flow_window['packet_count'].std()
            
            row['flow_byte_count_sum'] = flow_window['byte_count'].sum()
            row['flow_byte_count_mean'] = flow_window['byte_count'].mean()
            row['flow_byte_count_std'] = flow_window['byte_count'].std()
            
            row['flow_duration_sec_mean'] = flow_window['duration_sec'].mean()
            row['flow_duration_sec_std'] = flow_window['duration_sec'].std()
        else:
            row['flow_packet_count_sum'] = 0
            row['flow_packet_count_mean'] = 0
            row['flow_packet_count_std'] = 0
            
            row['flow_byte_count_sum'] = 0
            row['flow_byte_count_mean'] = 0
            row['flow_byte_count_std'] = 0
            
            row['flow_duration_sec_mean'] = 0
            row['flow_duration_sec_std'] = 0
        
        # Port statistics aggregation
        if len(port_window) > 0:
            row['port_rx_packets_sum'] = port_window['rx_packets'].sum()
            row['port_rx_packets_mean'] = port_window['rx_packets'].mean()
            row['port_rx_packets_std'] = port_window['rx_packets'].std()
            
            row['port_tx_packets_sum'] = port_window['tx_packets'].sum()
            row['port_tx_packets_mean'] = port_window['tx_packets'].mean()
            row['port_tx_packets_std'] = port_window['tx_packets'].std()
            
            row['port_rx_bytes_sum'] = port_window['rx_bytes'].sum()
            row['port_rx_bytes_mean'] = port_window['rx_bytes'].mean()
            row['port_rx_bytes_std'] = port_window['rx_bytes'].std()
            
            row['port_tx_bytes_sum'] = port_window['tx_bytes'].sum()
            row['port_tx_bytes_mean'] = port_window['tx_bytes'].mean()
            row['port_tx_bytes_std'] = port_window['tx_bytes'].std()
            
            row['port_rx_dropped_sum'] = port_window['rx_dropped'].sum()
            row['port_rx_dropped_mean'] = port_window['rx_dropped'].mean()
            row['port_rx_dropped_std'] = port_window['tx_dropped'].std()
            
            row['port_tx_errors_sum'] = port_window['tx_errors'].sum()
            row['port_tx_errors_mean'] = port_window['tx_errors'].mean()
            row['port_tx_errors_std'] = port_window['tx_errors'].std()
        else:
            for metric in ['rx_packets', 'tx_packets', 'rx_bytes', 'tx_bytes', 'rx_dropped', 'tx_dropped', 'rx_errors', 'tx_errors']:
                row[f'port_{metric}_sum'] = 0
                if metric in ['rx_packets', 'tx_packets', 'rx_bytes', 'tx_bytes']:
                    row[f'port_{metric}_mean'] = 0
                    row[f'port_{metric}_std'] = 0
        
        # Table statistics aggregation
        if len(table_window) > 0:
            row['table_active_count_sum'] = table_window['active_count'].sum()
            row['table_active_count_mean'] = table_window['active_count'].mean()
            row['table_active_count_std'] = table_window['active_count'].std()

            row['table_lookup_count_sum'] = table_window['lookup_count'].sum()
            row['table_lookup_count_mean'] = table_window['lookup_count'].mean()
            row['table_lookup_count_std'] = table_window['lookup_count'].std()
            
            row['table_matched_count_sum'] = table_window['matched_count'].sum()
            row['table_matched_count_mean'] = table_window['matched_count'].mean()
            row['table_matched_count_std'] = table_window['matched_count'].std()
        else:
            for metric in ['active_count', 'lookup_count', 'matched_count']:
                row[f'table_{metric}_sum'] = 0
                row[f'table_{metric}_mean'] = 0
                row[f'table_{metric}_std'] = 0

        # Protocol flow counts
        protocol_counts = flow_window['protocol'].value_counts()
        row['no_of_tcp_flows'] = protocol_counts.get('tcp', 0)
        row['no_of_udp_flows'] = protocol_counts.get('udp', 0)
        row['no_of_icmp_flows'] = protocol_counts.get('icmp', 0)
        row['no_of_other_flows'] = protocol_counts.get('other', 0)
        row['no_of_total_flows'] = len(flow_window)

        # Unique source/destination addresses and ports
        row['no_of_unique_src_addresses'] = flow_window['src_ip'].dropna().nunique()
        row['no_of_unique_dst_addresses'] = flow_window['dst_ip'].dropna().nunique()
        row['no_of_unique_src_ports'] = flow_window['src_port'].dropna().nunique()
        row['no_of_unique_dst_ports'] = flow_window['dst_port'].dropna().nunique()

        # Flow ratios
        row['tcp_to_udp_ratio'] = row['no_of_tcp_flows'] / (row['no_of_udp_flows'] + 1)
        row['flow_packets_per_flow'] = row['flow_packet_count_sum'] / (row['no_of_total_flows'] + 1)

        # Asymmetry ratios
        row['unique_src_to_dst_address_ratio'] = row['no_of_unique_src_addresses'] / (row['no_of_unique_dst_addresses'] + 1)
        row['unique_src_to_dst_port_ratio'] = row['no_of_unique_src_ports'] / (row['no_of_unique_dst_ports'] + 1)
        
        # Last 10 seconds statistics
        time_10s_ago = time_bucket - timedelta(seconds=10)
        flow_10s = flow_df[(flow_df['time_bucket'] > time_10s_ago) & (flow_df['time_bucket'] <= time_bucket)]
        port_10s = port_df[(port_df['time_bucket'] > time_10s_ago) & (port_df['time_bucket'] <= time_bucket)]
        
        row['flow_packet_count_10s_mean'] = flow_10s['packet_count'].mean() if len(flow_10s) > 0 else 0
        row['flow_byte_count_10s_mean'] = flow_10s['byte_count'].mean() if len(flow_10s) > 0 else 0
        row['port_rx_packets_10s_mean'] = port_10s['rx_packets'].mean() if len(port_10s) > 0 else 0
        row['port_tx_packets_10s_mean'] = port_10s['tx_packets'].mean() if len(port_10s) > 0 else 0
        row['total_flows_10s'] = len(flow_10s)
        
        # One-hot encoding for attack types
        # 0 = normal, 1 = attack present
        attack_types_present = attack_map.get(time_bucket, set())
        row['attack_syn'] = 1 if 'syn' in attack_types_present else 0
        row['attack_udp'] = 1 if 'udp' in attack_types_present else 0
        row['attack_icmp'] = 1 if 'icmp' in attack_types_present else 0
        
        # General attack indicator (for compatibility)
        row['attack'] = 1 if len(attack_types_present) > 0 else 0
        
        dataset_rows.append(row)
    
    print("Creating final dataset...")
    dataset = pd.DataFrame(dataset_rows)
    
    # Calculate differences from previous timestamp
    print("Calculating temporal differences...")
    diff_columns = [
        'flow_packet_count_sum', 
        'flow_byte_count_sum',
        'port_rx_packets_sum', 
        'port_tx_packets_sum',
        'port_rx_bytes_sum', 
        'port_tx_bytes_sum',
        'no_of_total_flows', 
        'no_of_tcp_flows',
        'no_of_udp_flows',
        'no_of_icmp_flows',
        'no_of_other_flows',
        'no_of_unique_src_addresses',
    ]
    
    for col in diff_columns:
        if col in dataset.columns:
            dataset[f'{col}_diff'] = dataset[col].diff().fillna(0)
    
    dataset = dataset.fillna(0)
    
    print(f"Dataset created with {len(dataset)} rows and {len(dataset.columns)} features")
    print(f"\nAttack distribution:")
    print(f"Normal samples: {(dataset['attack'] == 0).sum()}")
    print(f"SYN attack samples: {dataset['attack_syn'].sum()}")
    print(f"UDP attack samples: {dataset['attack_udp'].sum()}")
    print(f"ICMP attack samples: {dataset['attack_icmp'].sum()}")
    print(f"Total attack samples: {dataset['attack'].sum()}")
    
    return dataset

In [142]:
dataset = create_ml_dataset(
    flow_stats_path,
    port_stats_path,
    table_stats_path,
    alerts_path
)
    
dataset.to_csv(output_path, index=False)
print(f"\nDataset saved to {output_path}")

Parsing alerts...
Alerts before filtering: 155678
Alerts after filtering: 155678
Attack type distribution:
attack_type
syn     115472
udp      22553
icmp     17653
Name: count, dtype: int64
Parsing timestamps...

Filtering data to time range: 2025-10-17 15:25:46 to 2025-10-17 16:26:15
Rows after filtering - Flow: 1502344, Port: 14928, Table: 9946, Alerts: 155678

Time buckets with attacks: 1805
Extracting flow match fields...
Processing 3354 time windows...
Processing 0/3354...
Processing 100/3354...
Processing 200/3354...
Processing 300/3354...
Processing 400/3354...
Processing 500/3354...
Processing 600/3354...
Processing 700/3354...
Processing 800/3354...
Processing 900/3354...
Processing 1000/3354...
Processing 1100/3354...
Processing 1200/3354...
Processing 1300/3354...
Processing 1400/3354...
Processing 1500/3354...
Processing 1600/3354...
Processing 1700/3354...
Processing 1800/3354...
Processing 1900/3354...
Processing 2000/3354...
Processing 2100/3354...
Processing 2200/3354..

In [143]:
print("\nDataset Info:")
print(dataset.info())
print("\nFirst few rows:")
print(dataset.head())
print("\nFeature statistics:")
print(dataset.describe())
print("\nClass distribution:")
print(dataset['attack'].value_counts())
print("\nAttack type distribution:")
print(f"SYN attacks: {dataset['attack_syn'].sum()}")
print(f"UDP attacks: {dataset['attack_udp'].sum()}")
print(f"ICMP attacks: {dataset['attack_icmp'].sum()}")


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3354 entries, 0 to 3353
Data columns (total 72 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        3354 non-null   datetime64[ns]
 1   flow_packet_count_sum            3354 non-null   int64         
 2   flow_packet_count_mean           3354 non-null   float64       
 3   flow_packet_count_std            3354 non-null   float64       
 4   flow_byte_count_sum              3354 non-null   int64         
 5   flow_byte_count_mean             3354 non-null   float64       
 6   flow_byte_count_std              3354 non-null   float64       
 7   flow_duration_sec_mean           3354 non-null   float64       
 8   flow_duration_sec_std            3354 non-null   float64       
 9   port_rx_packets_sum              3354 non-null   int64         
 10  port_rx_packets_mean             3354 non-nul

In [144]:
print("\nAttack distribution by type:")
attack_samples = dataset[dataset['attack'] == 1]
print(f"\nTotal attack samples: {len(attack_samples)}")
print(f"SYN attack samples: {dataset['attack_syn'].sum()}")
print(f"UDP attack samples: {dataset['attack_udp'].sum()}")
print(f"ICMP attack samples: {dataset['attack_icmp'].sum()}")

if len(attack_samples) > 0:
    print("\nFirst 10 attack rows:")
    print(attack_samples[['timestamp', 'no_of_total_flows', 'no_of_tcp_flows', 'no_of_udp_flows',
                          'no_of_icmp_flows', 'attack_syn', 'attack_udp', 'attack_icmp']].head(10))

print("\nFlow count distribution:")
print(dataset['no_of_total_flows'].describe())

high_flow = dataset[dataset['no_of_total_flows'] > 100]
print(f"\nRows with >100 flows: {len(high_flow)}")
print(high_flow[['timestamp', 'no_of_total_flows', 'no_of_tcp_flows', 
                 'attack_syn', 'attack_udp', 'attack_icmp']].head(10))


Attack distribution by type:

Total attack samples: 1577
SYN attack samples: 634
UDP attack samples: 599
ICMP attack samples: 344

First 10 attack rows:
            timestamp  no_of_total_flows  no_of_tcp_flows  no_of_udp_flows  \
0 2025-10-17 15:25:46                  4                0                0   
1 2025-10-17 15:25:48                 77               75                0   
2 2025-10-17 15:25:49                218              216                0   
3 2025-10-17 15:25:50                850              848                0   
4 2025-10-17 15:25:51               1335             1333                0   
5 2025-10-17 15:25:56                890              888                0   
6 2025-10-17 15:25:57                966              962                0   
7 2025-10-17 15:25:58               1549             1545                0   
8 2025-10-17 15:25:59                491              489                0   
9 2025-10-17 15:26:00               1619             1617         

In [145]:
print("\nChecking feature diversity:")
print(f"Unique no_of_total_flows values: {dataset['no_of_total_flows'].nunique()}")
print(f"Unique no_of_tcp_flows values: {dataset['no_of_tcp_flows'].nunique()}")
print(f"Unique no_of_udp_flows values: {dataset['no_of_udp_flows'].nunique()}")
print(f"Unique no_of_icmp_flows values: {dataset['no_of_icmp_flows'].nunique()}")

print("\nPacket and byte statistics:")
print(dataset[['flow_packet_count_sum', 'flow_byte_count_sum', 
               'port_rx_packets_sum', 'port_tx_packets_sum']].describe())

print("\nAttack vs Normal comparison:")
print("\nNormal samples (mean):")
print(dataset[dataset['attack']==0][['no_of_total_flows', 'no_of_tcp_flows', 
                                      'no_of_udp_flows', 'no_of_icmp_flows',
                                      'flow_packet_count_sum']].mean())
print("\nSYN attack samples (mean):")
syn_samples = dataset[dataset['attack_syn']==1]
if len(syn_samples) > 0:
    print(syn_samples[['no_of_total_flows', 'no_of_tcp_flows', 
                       'no_of_udp_flows', 'no_of_icmp_flows',
                       'flow_packet_count_sum']].mean())

print("\nUDP attack samples (mean):")
udp_samples = dataset[dataset['attack_udp']==1]
if len(udp_samples) > 0:
    print(udp_samples[['no_of_total_flows', 'no_of_tcp_flows', 
                       'no_of_udp_flows', 'no_of_icmp_flows',
                       'flow_packet_count_sum']].mean())

print("\nICMP attack samples (mean):")
icmp_samples = dataset[dataset['attack_icmp']==1]
if len(icmp_samples) > 0:
    print(icmp_samples[['no_of_total_flows', 'no_of_tcp_flows', 
                        'no_of_udp_flows', 'no_of_icmp_flows',
                        'flow_packet_count_sum']].mean())


Checking feature diversity:
Unique no_of_total_flows values: 1066
Unique no_of_tcp_flows values: 698
Unique no_of_udp_flows values: 466
Unique no_of_icmp_flows values: 241

Packet and byte statistics:
       flow_packet_count_sum  flow_byte_count_sum  port_rx_packets_sum  \
count           3.354000e+03         3.354000e+03         3.354000e+03   
mean            2.078132e+06         7.673042e+08         1.478462e+06   
std             3.183594e+06         1.132390e+09         1.968350e+06   
min             0.000000e+00         0.000000e+00         0.000000e+00   
25%             3.443692e+05         1.219276e+08         3.508840e+05   
50%             8.805955e+05         3.551053e+08         9.170675e+05   
75%             3.383001e+06         1.198733e+09         2.204329e+06   
max             6.745981e+07         2.446180e+10         4.011766e+07   

       port_tx_packets_sum  
count         3.354000e+03  
mean          8.464079e+05  
std           7.387333e+05  
min           0

In [146]:
def parse_attack_windows(log_file_path):
    """
    Parse attack windows from traffic_simulation.log
    Returns a dict mapping attack window name (as in the log) to list of windows:
      { 'tcp_syn_flood': [(start, end), ...], ... }
    The function lower-cases the window name so it's stable.
    """
    attack_windows = {}
    current_attacks = {}

    with open(log_file_path, 'r') as f:
        for line in f:
            line = line.strip()

            start_match = re.search(r'\[(.*?)\]\s+!!! Starting ATTACK window ([\w_]+)', line, re.IGNORECASE)
            if start_match:
                timestamp = pd.to_datetime(start_match.group(1))
                attack_type = start_match.group(2).lower().strip()
                current_attacks[attack_type] = timestamp
                if attack_type not in attack_windows:
                    attack_windows[attack_type] = []
                continue

            end_match = re.search(r'\[(.*?)\]\s+!!! Ending ATTACK window ([\w_]+)', line, re.IGNORECASE)
            if end_match:
                timestamp = pd.to_datetime(end_match.group(1))
                attack_type = end_match.group(2).lower().strip()
                if attack_type in current_attacks:
                    start_time = current_attacks[attack_type]
                    attack_windows.setdefault(attack_type, []).append((start_time, timestamp))
                    del current_attacks[attack_type]
                continue

    return attack_windows

In [147]:
def compute_snort_accuracy_per_attack_type(alerts_df, attack_windows):
    """
    Compute detection statistics per second for each attack type.
    attack_windows: dict keyed by attack window name (e.g., 'tcp_syn_flood') with list of (start,end) tuples.
    alerts_df: DataFrame with columns ['timestamp', 'attack_type'] where attack_type is 'syn','udp','icmp',...
    Returns dict with stats for each window key from attack_windows.
    """
    results = {}

    # Precompute floor('1s') timestamps for all alerts grouped by alert label
    alerts_df = alerts_df.copy()
    alerts_df['second'] = alerts_df['timestamp'].dt.floor('1s')
    alerts_by_label = {}
    for label, grp in alerts_df.groupby('attack_type'):
        # convert to set of Timestamps for O(1) membership
        alerts_by_label[label.lower().strip()] = set(grp['second'].tolist())

    for window_name, windows in attack_windows.items():
        # map the verbose window name to an alert label (syn/udp/icmp)
        alert_label = _map_window_name_to_alert_label(window_name)
        if alert_label is None:
            # unknown/unsupported window name -> record zero coverage
            results[window_name] = {
                'total_attack_seconds': 0,
                'detected_seconds': 0,
                'missing_seconds': [],
                'coverage': 0.0,
                'mapped_label': None
            }
            continue

        alerts_seconds = alerts_by_label.get(alert_label, set())

        total_attack_seconds = 0
        detected_seconds = 0
        missing_seconds = []

        for start, end in windows:
            # ensure start/end are Timestamps (they usually are from parse_attack_windows)
            start_ts = pd.to_datetime(start)
            end_ts = pd.to_datetime(end)
            # iterate per-second (inclusive start/end). Note: you might want to exclude end if end marks first non-attack second.
            seconds_in_window = pd.date_range(start=start_ts.floor('1s'), end=end_ts.floor('1s'), freq='1s')
            total_attack_seconds += len(seconds_in_window)

            for sec in seconds_in_window:
                if sec in alerts_seconds:
                    detected_seconds += 1
                else:
                    missing_seconds.append(sec)

        coverage = (detected_seconds / total_attack_seconds) if total_attack_seconds > 0 else 0.0

        results[window_name] = {
            'total_attack_seconds': total_attack_seconds,
            'detected_seconds': detected_seconds,
            'missing_seconds': missing_seconds,
            'coverage': coverage,
            'mapped_label': alert_label
        }

    return results

In [148]:
def _map_window_name_to_alert_label(window_name: str):
    """
    Map full attack window names (from traffic_simulation.log) to the short alert labels
    used in alerts_df produced by parse_alert_line().
    Examples:
      'tcp_syn_flood' -> 'syn'
      'udp_flood'     -> 'udp'
      'icmp_flood'    -> 'icmp'
    The mapping is permissive: it searches substrings.
    """
    if not isinstance(window_name, str):
        return None
    wn = window_name.lower()
    if 'syn' in wn:
        return 'syn'
    if 'udp' in wn:
        return 'udp'
    if 'icmp' in wn or 'ping' in wn:
        return 'icmp'
    return None

In [149]:
# Parse attack windows from log
attack_windows = parse_attack_windows(log_file_path)

# Reload alerts with attack types
with open(alerts_path, 'r') as f:
    alert_lines = [line.strip() for line in f if line.strip()]

parsed_alerts = [parse_alert_line(line) for line in alert_lines]
alerts_df = pd.DataFrame(parsed_alerts, columns=['timestamp', 'attack_type'])
alerts_df = alerts_df[alerts_df['timestamp'].notna() & alerts_df['attack_type'].notna()]
alerts_df = alerts_df[(alerts_df['timestamp'] >= start_time) & (alerts_df['timestamp'] <= end_time)]

# Compute accuracy per attack type
per_type_stats = compute_snort_accuracy_per_attack_type(alerts_df, attack_windows)

In [150]:
print("\nDetection Statistics by Attack Type:\n")
total_attacks_seconds_all = 0
for attack_type in ['tcp_syn_flood', 'udp_flood', 'icmp_flood']:
    stats = per_type_stats[attack_type]
    print(f"{attack_type.upper()} Attack:")
    print(f"  Total attack seconds: {stats['total_attack_seconds']}")
    total_attacks_seconds_all+=stats['total_attack_seconds']
    print(f"  Detected seconds: {stats['detected_seconds']}")
    print(f"  Missing seconds: {len(stats['missing_seconds'])}")
    print(f"  Coverage: {stats['coverage'] * 100:.2f}%\n")
print(f"Total attacks seconds before manual labeling: {total_attacks_seconds_all}")


Detection Statistics by Attack Type:

TCP_SYN_FLOOD Attack:
  Total attack seconds: 906
  Detected seconds: 785
  Missing seconds: 121
  Coverage: 86.64%

UDP_FLOOD Attack:
  Total attack seconds: 704
  Detected seconds: 644
  Missing seconds: 60
  Coverage: 91.48%

ICMP_FLOOD Attack:
  Total attack seconds: 425
  Detected seconds: 377
  Missing seconds: 48
  Coverage: 88.71%

Total attacks seconds before manual labeling: 2035


In [151]:
def apply_manual_labeling(dataset, attack_windows):
    """
    Apply manual attack labeling based on actual attack windows from the log.
    This corrects labels for seconds where Snort missed alerts but attacks were actually happening.
    Uses one-hot encoding: each sample belongs to only one attack type.
    For overlapping attack windows, uses temporal context (previous/next rows) to determine type.
    
    IMPORTANT: This function only ADDS missing labels, it doesn't remove existing Snort detections.
    
    Args:
        dataset: The ML dataset DataFrame with timestamp and attack columns
        attack_windows: Dict mapping attack type to list of (start, end) tuples
    
    Returns:
        New corrected dataset (original dataset is not modified)
    """
    print("\nApplying Manual Attack Labeling\n")
    
    # Make a copy to avoid modifying original
    dataset_corrected = dataset.copy()
    
    # Track corrections
    corrections = {
        'syn': 0,
        'udp': 0,
        'icmp': 0,
        'total': 0
    }
    
    # Store original labels for comparison
    original_attack = dataset['attack'].sum()
    original_syn = dataset['attack_syn'].sum()
    original_udp = dataset['attack_udp'].sum()
    original_icmp = dataset['attack_icmp'].sum()
    
    # Create a mapping of timestamp to possible attack types
    timestamp_possible_attacks = {}
    
    for window_name, windows in attack_windows.items():
        attack_label = _map_window_name_to_alert_label(window_name)
        if attack_label is None:
            print(f"Warning: Unknown window type '{window_name}', skipping")
            continue
        
        for start, end in windows:
            start_ts = pd.to_datetime(start).floor('1s')
            end_ts = pd.to_datetime(end).floor('1s')
            seconds_in_window = pd.date_range(start=start_ts, end=end_ts, freq='1s')
            
            for sec in seconds_in_window:
                if sec not in timestamp_possible_attacks:
                    timestamp_possible_attacks[sec] = set()
                timestamp_possible_attacks[sec].add(attack_label)
    
    print(f"Total unique attack timestamps mapped: {len(timestamp_possible_attacks)}")
    
    # Count overlaps
    overlap_count = sum(1 for attacks in timestamp_possible_attacks.values() if len(attacks) > 1)
    print(f"Timestamps with overlapping attack windows: {overlap_count}")
    
    # Apply labels - only for rows that are NOT already labeled or need correction
    for idx in range(len(dataset_corrected)):
        ts = pd.to_datetime(dataset_corrected.iloc[idx]['timestamp']).floor('1s')
        
        if ts in timestamp_possible_attacks:
            possible_attacks = timestamp_possible_attacks[ts]
            
            # Check if this row already has an attack label
            current_syn = dataset_corrected.iloc[idx]['attack_syn']
            current_udp = dataset_corrected.iloc[idx]['attack_udp']
            current_icmp = dataset_corrected.iloc[idx]['attack_icmp']
            current_attack = dataset_corrected.iloc[idx]['attack']
            
            # If already labeled, check if it matches one of the possible attacks
            if current_attack == 1:
                current_type = None
                if current_syn == 1:
                    current_type = 'syn'
                elif current_udp == 1:
                    current_type = 'udp'
                elif current_icmp == 1:
                    current_type = 'icmp'
                
                # If current label is valid (in possible attacks), keep it
                if current_type and current_type in possible_attacks:
                    continue  # Keep existing label
                
                # If current label is not in possible attacks but there are possible attacks,
                # we need to correct it (handle one-hot violation from Snort)
                if current_type and current_type not in possible_attacks:
                    # This shouldn't happen often, but handle it
                    # Reset and relabel based on context
                    dataset_corrected.at[idx, 'attack_syn'] = 0
                    dataset_corrected.at[idx, 'attack_udp'] = 0
                    dataset_corrected.at[idx, 'attack_icmp'] = 0
            
            # If not labeled or needs correction, determine the attack type
            if current_attack == 0 or (current_syn + current_udp + current_icmp) == 0:
                # Determine attack type
                if len(possible_attacks) == 1:
                    attack_type = list(possible_attacks)[0]
                else:
                    # Multiple possible attacks - use temporal context
                    attack_type = resolve_attack_type_by_context(
                        dataset_corrected, idx, possible_attacks
                    )
                
                # Apply the label
                dataset_corrected.at[idx, 'attack'] = 1
                dataset_corrected.at[idx, f'attack_{attack_type}'] = 1
                
                # Track correction
                if dataset.iloc[idx]['attack'] == 0:
                    corrections[attack_type] += 1
                    corrections['total'] += 1
    
    # Handle any remaining one-hot violations (samples with multiple attack flags)
    # This can happen if Snort detected multiple attack types for the same timestamp
    print("\nChecking for one-hot violations...")
    for idx in range(len(dataset_corrected)):
        syn = dataset_corrected.iloc[idx]['attack_syn']
        udp = dataset_corrected.iloc[idx]['attack_udp']
        icmp = dataset_corrected.iloc[idx]['attack_icmp']
        
        if (syn + udp + icmp) > 1:
            # Violation detected - use temporal context to resolve
            ts = pd.to_datetime(dataset_corrected.iloc[idx]['timestamp']).floor('1s')
            possible = set()
            if syn == 1:
                possible.add('syn')
            if udp == 1:
                possible.add('udp')
            if icmp == 1:
                possible.add('icmp')
            
            # Reset all
            dataset_corrected.at[idx, 'attack_syn'] = 0
            dataset_corrected.at[idx, 'attack_udp'] = 0
            dataset_corrected.at[idx, 'attack_icmp'] = 0
            
            # Resolve using context
            attack_type = resolve_attack_type_by_context(dataset_corrected, idx, possible)
            dataset_corrected.at[idx, f'attack_{attack_type}'] = 1
            dataset_corrected.at[idx, 'attack'] = 1
    
    # Print summary
    print(f"\nLabeling correction summary\n")
    print(f"Original labels (Snort-based):")
    print(f"  Total attack samples: {original_attack}")
    print(f"  SYN attack samples: {original_syn}")
    print(f"  UDP attack samples: {original_udp}")
    print(f"  ICMP attack samples: {original_icmp}")
    
    print(f"\nCorrections applied:")
    print(f"  SYN corrections: {corrections['syn']}")
    print(f"  UDP corrections: {corrections['udp']}")
    print(f"  ICMP corrections: {corrections['icmp']}")
    print(f"  Total corrections: {corrections['total']}")
    
    print(f"\nFinal labels (after manual correction):")
    print(f"  Total attack samples: {dataset_corrected['attack'].sum()}")
    print(f"  SYN attack samples: {dataset_corrected['attack_syn'].sum()}")
    print(f"  UDP attack samples: {dataset_corrected['attack_udp'].sum()}")
    print(f"  ICMP attack samples: {dataset_corrected['attack_icmp'].sum()}")
    
    print(f"\nNormal samples: {(dataset_corrected['attack'] == 0).sum()}")
    
    # Verify one-hot encoding
    multi_label_count = ((dataset_corrected['attack_syn'] + 
                          dataset_corrected['attack_udp'] + 
                          dataset_corrected['attack_icmp']) > 1).sum()
    
    if multi_label_count > 0:
        print(f"\nWARNING: Found {multi_label_count} samples with multiple attack labels!")
    else:
        print(f"\nOne-hot encoding verified: All samples have at most one attack type")
    
    return dataset_corrected, corrections

In [152]:
def resolve_attack_type_by_context(df, idx, possible_attacks):
    """
    Resolve which attack type to assign when multiple attacks overlap at a timestamp.
    Uses temporal context: if previous and next rows have the same attack type, use that.
    
    Args:
        df: DataFrame with attack columns
        idx: Current row index
        possible_attacks: Set of possible attack types for this timestamp
    
    Returns:
        The resolved attack type (str: 'syn', 'udp', or 'icmp')
    """
    # Look at previous and next rows
    prev_attack = None
    next_attack = None
    
    # Check previous row
    if idx > 0:
        prev_row = df.iloc[idx - 1]
        if prev_row['attack_syn'] == 1:
            prev_attack = 'syn'
        elif prev_row['attack_udp'] == 1:
            prev_attack = 'udp'
        elif prev_row['attack_icmp'] == 1:
            prev_attack = 'icmp'
    
    # Check next row
    if idx < len(df) - 1:
        next_row = df.iloc[idx + 1]
        if next_row['attack_syn'] == 1:
            next_attack = 'syn'
        elif next_row['attack_udp'] == 1:
            next_attack = 'udp'
        elif next_row['attack_icmp'] == 1:
            next_attack = 'icmp'
    
    # If previous and next are the same attack type, and it's in possible attacks, use it
    if prev_attack and prev_attack == next_attack and prev_attack in possible_attacks:
        return prev_attack
    
    # If only previous matches and it's in possible attacks, use it
    if prev_attack and prev_attack in possible_attacks:
        return prev_attack
    
    # If only next matches and it's in possible attacks, use it
    if next_attack and next_attack in possible_attacks:
        return next_attack
    
    # Fallback: use priority (this should rarely happen)
    # Priority: syn > udp > icmp
    priority_order = ['syn', 'udp', 'icmp']
    for attack_type in priority_order:
        if attack_type in possible_attacks:
            return attack_type
    
    # Should never reach here, but return first available as last resort
    return list(possible_attacks)[0]

In [153]:
# Apply manual labeling to create corrected dataset (original dataset remains unchanged)
dataset_corrected, correction_stats = apply_manual_labeling(dataset, attack_windows)


Applying Manual Attack Labeling

Total unique attack timestamps mapped: 2031
Timestamps with overlapping attack windows: 4

Checking for one-hot violations...

=== Labeling Correction Summary ===

Original labels (Snort-based):
  Total attack samples: 1577
  SYN attack samples: 634
  UDP attack samples: 599
  ICMP attack samples: 344

Corrections applied:
  SYN corrections: 100
  UDP corrections: 52
  ICMP corrections: 42
  Total corrections: 194

Final labels (after manual correction):
  Total attack samples: 1771
  SYN attack samples: 734
  UDP attack samples: 651
  ICMP attack samples: 386

Normal samples: 1583

One-hot encoding verified: All samples have at most one attack type


In [154]:
# Save the corrected dataset with -corrected suffix
corrected_output_path = output_path.replace('.csv', '-corrected.csv')
dataset_corrected.to_csv(corrected_output_path, index=False)
print(f"\nCorrected dataset saved to {corrected_output_path}")
print(f"Original dataset remains at {output_path}")


Corrected dataset saved to ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks-corrected.csv
Original dataset remains at ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks.csv


In [155]:
# Show comparison between original and corrected datasets
print("\nDataset Comparison\n")
print(f"Original dataset: {output_path}")
print(f"  Rows: {len(dataset)}")
print(f"  Attack samples: {dataset['attack'].sum()}")
print(f"  Normal samples: {(dataset['attack'] == 0).sum()}")

print(f"\nCorrected dataset: {corrected_output_path}")
print(f"  Rows: {len(dataset_corrected)}")
print(f"  Attack samples: {dataset_corrected['attack'].sum()}")
print(f"  Normal samples: {(dataset_corrected['attack'] == 0).sum()}")

print(f"\nDifference:")
print(f"  Additional attack samples: {dataset_corrected['attack'].sum() - dataset['attack'].sum()}")
print(f"  Reduced normal samples: {(dataset['attack'] == 0).sum() - (dataset_corrected['attack'] == 0).sum()}")


Dataset Comparison

Original dataset: ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks.csv
  Rows: 3354
  Attack samples: 1577
  Normal samples: 1777

Corrected dataset: ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks-corrected.csv
  Rows: 3354
  Attack samples: 1771
  Normal samples: 1583

Difference:
  Additional attack samples: 194
  Reduced normal samples: 194


In [156]:
print("Sample of corrected rows:\n")
for attack_type in ['syn', 'udp', 'icmp']:
    col = f'attack_{attack_type}'
    
    # Find rows that were corrected
    corrected_mask = (dataset_corrected[col] == 1) & (dataset[col] == 0)
    corrected_samples = dataset_corrected[corrected_mask]
    
    if len(corrected_samples) > 0:
        print(f"\n{attack_type.upper()} - First 5 corrected samples:")
        print(corrected_samples[['timestamp', 'no_of_total_flows', 'no_of_tcp_flows', 
                                  'no_of_udp_flows', 'no_of_icmp_flows', 
                                  'attack_syn', 'attack_udp', 'attack_icmp']].head())
    else:
        print(f"\n{attack_type.upper()} - No corrections needed (100% Snort coverage)")

Sample of corrected rows:


SYN - First 5 corrected samples:
             timestamp  no_of_total_flows  no_of_tcp_flows  no_of_udp_flows  \
11 2025-10-17 15:26:06                 34               32                0   
12 2025-10-17 15:26:07                724              718                0   
69 2025-10-17 15:27:10                614              611                0   
70 2025-10-17 15:27:12                 77               75                0   
71 2025-10-17 15:27:13               2274             2270                0   

    no_of_icmp_flows  attack_syn  attack_udp  attack_icmp  
11                 0           1           0            0  
12                 0           1           0            0  
69                 0           1           0            0  
70                 0           1           0            0  
71                 0           1           0            0  

UDP - First 5 corrected samples:
              timestamp  no_of_total_flows  no_of_tcp_flows  no_of_udp

In [157]:
print("\nFinal Statistics:\n")

print("ORIGINAL DATASET (Snort-based labels):")
print(f"  File: {output_path}")
print(f"  Total rows: {len(dataset)}")
print(f"  Normal: {(dataset['attack'] == 0).sum()} ({(dataset['attack'] == 0).sum() / len(dataset) * 100:.2f}%)")
print(f"  Attack: {dataset['attack'].sum()} ({dataset['attack'].sum() / len(dataset) * 100:.2f}%)")
print(f"    - SYN: {dataset['attack_syn'].sum()}")
print(f"    - UDP: {dataset['attack_udp'].sum()}")
print(f"    - ICMP: {dataset['attack_icmp'].sum()}")

print("\nCORRECTED DATASET (Ground truth from logs):")
print(f"  File: {corrected_output_path}")
print(f"  Total rows: {len(dataset_corrected)}")
print(f"  Normal: {(dataset_corrected['attack'] == 0).sum()} ({(dataset_corrected['attack'] == 0).sum() / len(dataset_corrected) * 100:.2f}%)")
print(f"  Attack: {dataset_corrected['attack'].sum()} ({dataset_corrected['attack'].sum() / len(dataset_corrected) * 100:.2f}%)")
print(f"    - SYN: {dataset_corrected['attack_syn'].sum()}")
print(f"    - UDP: {dataset_corrected['attack_udp'].sum()}")
print(f"    - ICMP: {dataset_corrected['attack_icmp'].sum()}")


Final Statistics:

ORIGINAL DATASET (Snort-based labels):
  File: ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks.csv
  Total rows: 3354
  Normal: 1777 (52.98%)
  Attack: 1577 (47.02%)
    - SYN: 634
    - UDP: 599
    - ICMP: 344

CORRECTED DATASET (Ground truth from logs):
  File: ml-dataset-50-50-idle-timeout-2-17-10-2025-multiple-attacks-corrected.csv
  Total rows: 3354
  Normal: 1583 (47.20%)
  Attack: 1771 (52.80%)
    - SYN: 734
    - UDP: 651
    - ICMP: 386


In [160]:
def verify_manual_labeling(dataset, dataset_corrected, attack_windows, per_type_stats):
    """
    Verify the manual labeling results against expected values.
    """
    print("\nManual Labeling Verification\n")
    
    # Calculate expected corrections from per_type_stats
    expected_corrections = {}
    total_expected = 0
    
    for attack_type in ['tcp_syn_flood', 'udp_flood', 'icmp_flood']:
        stats = per_type_stats[attack_type]
        missing = len(stats['missing_seconds'])
        attack_label = _map_window_name_to_alert_label(attack_type)
        expected_corrections[attack_label] = missing
        total_expected += missing
    
    print("Expected corrections (from missing seconds):")
    print(f"  SYN: {expected_corrections['syn']}")
    print(f"  UDP: {expected_corrections['udp']}")
    print(f"  ICMP: {expected_corrections['icmp']}")
    print(f"  Total: {total_expected}")
    
    # Calculate actual corrections
    actual_corrections = {}
    actual_corrections['syn'] = (dataset_corrected['attack_syn'] == 1).sum() - (dataset['attack_syn'] == 1).sum()
    actual_corrections['udp'] = (dataset_corrected['attack_udp'] == 1).sum() - (dataset['attack_udp'] == 1).sum()
    actual_corrections['icmp'] = (dataset_corrected['attack_icmp'] == 1).sum() - (dataset['attack_icmp'] == 1).sum()
    total_actual = sum(actual_corrections.values())
    
    print("\nActual corrections applied:")
    print(f"  SYN: {actual_corrections['syn']}")
    print(f"  UDP: {actual_corrections['udp']}")
    print(f"  ICMP: {actual_corrections['icmp']}")
    print(f"  Total: {total_actual}")
    
    print("\nDifference (Expected - Actual):")
    print(f"  SYN: {expected_corrections['syn'] - actual_corrections['syn']}")
    print(f"  UDP: {expected_corrections['udp'] - actual_corrections['udp']}")
    print(f"  ICMP: {expected_corrections['icmp'] - actual_corrections['icmp']}")
    print(f"  Total: {total_expected - total_actual}")
    
    print("\nChecking Missing Seconds Not in Dataset\n")
    
    dataset_timestamps = set(pd.to_datetime(dataset['timestamp']).dt.floor('1s'))
    
    for attack_type in ['tcp_syn_flood', 'udp_flood', 'icmp_flood']:
        stats = per_type_stats[attack_type]
        attack_label = _map_window_name_to_alert_label(attack_type)
        missing_seconds = stats['missing_seconds']
        
        # Check which missing seconds are not in dataset
        not_in_dataset = [ts for ts in missing_seconds if ts not in dataset_timestamps]
        
        print(f"{attack_type.upper()}:")
        print(f"  Missing seconds: {len(missing_seconds)}")
        print(f"  Missing seconds NOT in dataset: {len(not_in_dataset)}")
        print(f"  Missing seconds IN dataset (should be corrected): {len(missing_seconds) - len(not_in_dataset)}")
        
        if len(not_in_dataset) > 0:
            print(f"  Sample timestamps not in dataset: {not_in_dataset[:5]}")
    
    print("\nTotal Attack Samples Verification\n")
    
    total_attack_seconds = sum(per_type_stats[at]['total_attack_seconds'] 
                               for at in ['tcp_syn_flood', 'udp_flood', 'icmp_flood'])
    detected_seconds = sum(per_type_stats[at]['detected_seconds'] 
                          for at in ['tcp_syn_flood', 'udp_flood', 'icmp_flood'])
    
    print(f"Total attack seconds (from windows): {total_attack_seconds}")
    print(f"Detected by Snort: {detected_seconds}")
    print(f"Missing by Snort: {total_attack_seconds - detected_seconds}")
    
    # Count how many of these timestamps are in dataset
    all_attack_timestamps = set()
    for attack_type, windows in attack_windows.items():
        for start, end in windows:
            start_ts = pd.to_datetime(start).floor('1s')
            end_ts = pd.to_datetime(end).floor('1s')
            seconds_in_window = pd.date_range(start=start_ts, end=end_ts, freq='1s')
            all_attack_timestamps.update(seconds_in_window)
    
    attack_timestamps_in_dataset = [ts for ts in all_attack_timestamps if ts in dataset_timestamps]
    
    print(f"\nAttack timestamps in windows: {len(all_attack_timestamps)}")
    print(f"Attack timestamps in dataset: {len(attack_timestamps_in_dataset)}")
    print(f"Attack timestamps NOT in dataset: {len(all_attack_timestamps) - len(attack_timestamps_in_dataset)}")
    
    print(f"\nExpected final attack samples: {len(attack_timestamps_in_dataset)}")
    print(f"Actual final attack samples: {dataset_corrected['attack'].sum()}")
    print(f"Difference: {len(attack_timestamps_in_dataset) - dataset_corrected['attack'].sum()}")

verify_manual_labeling(dataset, dataset_corrected, attack_windows, per_type_stats)


Manual Labeling Verification

Expected corrections (from missing seconds):
  SYN: 121
  UDP: 60
  ICMP: 48
  Total: 229

Actual corrections applied:
  SYN: 100
  UDP: 52
  ICMP: 42
  Total: 194

Difference (Expected - Actual):
  SYN: 21
  UDP: 8
  ICMP: 6
  Total: 35

Checking Missing Seconds Not in Dataset

TCP_SYN_FLOOD:
  Missing seconds: 121
  Missing seconds NOT in dataset: 21
  Missing seconds IN dataset (should be corrected): 100
  Sample timestamps not in dataset: [Timestamp('2025-10-17 15:26:04'), Timestamp('2025-10-17 15:26:05'), Timestamp('2025-10-17 15:26:08'), Timestamp('2025-10-17 15:27:11'), Timestamp('2025-10-17 15:31:28')]
UDP_FLOOD:
  Missing seconds: 60
  Missing seconds NOT in dataset: 5
  Missing seconds IN dataset (should be corrected): 55
  Sample timestamps not in dataset: [Timestamp('2025-10-17 16:00:17'), Timestamp('2025-10-17 16:00:19'), Timestamp('2025-10-17 16:01:27'), Timestamp('2025-10-17 16:01:39'), Timestamp('2025-10-17 16:22:41')]
ICMP_FLOOD:
  Missin